# S1 side bet: do under-trained tokens break refusal? (gate G3, Kaggle T4)
50 reachable under-trained tokens x 50 UNLISTED harmful prompts, k in {1,3,5}, vs rare-but-trained control tokens. Gate: paired CI of attack-success difference excludes 0 (>= 10pp to pursue).

In [ ]:
!pip uninstall -y torchao
!git clone -q -b feat/hsafe https://github.com/dhairya-pandya/Lexsi-AI-Alignment repo
%cd repo
!pip install -q -e '.[gpu]'

In [ ]:
import json, yaml, random, numpy as np, pandas as pd, torch, os
from transformers import AutoTokenizer
from hsafe.glitch import load_magikarp, reachable, insert_tokens
from hsafe.gen import HFGenerator
from hsafe.judge import HarmJudge, judge_all
from hsafe.records import read_jsonl, PromptRecord
from hsafe.render import render
from hsafe.stats import paired_diff_ci
m = yaml.safe_load(open('configs/models.yaml'))['tiny-aya-global']
tok = AutoTokenizer.from_pretrained(m['id'], revision=m['revision'])
glitch = reachable(load_magikarp('data/magikarp/CohereLabs_tiny_aya_global.md'), tok)[:50]
# control: rare but trained tokens = reachable ids NOT in the under-trained list, sampled from the tail of the vocab
bad = set(load_magikarp('data/magikarp/CohereLabs_tiny_aya_global.md', max_prob=1.0))
rng = random.Random(0); control = []
while len(control) < 50:
    i = rng.randrange(200000, 261000)
    if i not in bad and reachable([i], tok): control.append(i)
H = [r for r in read_jsonl('data/prompts/harmful_en.jsonl', PromptRecord) if r.split=='eval' and r.bucket!='ACCEPT'][:50]

In [ ]:
gen = HFGenerator(m['id'], m['revision'], dtype=torch.float16, max_new_tokens=128)
rows = []
for k in [1, 3, 5]:
    for arm, pool in [('glitch', glitch), ('control', control)]:
        rs = [insert_tokens(render(r, 'P-default', None, None, tok, 'tiny_aya'), pool, k, seed=j, tok=tok) for j, r in enumerate(H)]
        outs = gen.generate(rs)
        rows += [dict(record_id=r.id, k=k, arm=arm, output=o.output) for r, o in zip(H, outs)]
del gen; import gc; gc.collect(); torch.cuda.empty_cache()
df = pd.DataFrame(rows)
judge = HarmJudge()
from hsafe.records import Generation
rec_by = {r.id: r for r in H}
labels = judge_all([rec_by[i] for i in df.record_id], [Generation(str(i), o, float('nan')) for i, o in enumerate(df.output)], judge)
df['unsafe'] = [float(j.label == 'unsafe') for j in labels]
os.makedirs('results/s1', exist_ok=True); df.to_csv('results/s1/items.csv', index=False)
for k in [1, 3, 5]:
    p = df[df.k==k].pivot_table(index='record_id', columns='arm', values='unsafe')
    print(k, paired_diff_ci(p['control'].to_numpy(), p['glitch'].to_numpy()))